In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded!")

In [ ]:
# Load Nigeria data
df = pd.read_csv('data/nigeria.csv')
df['Country'] = 'Nigeria'

print(f"Data loaded! Shape: {df.shape}")
print("\nFirst 5 rows:")
df.head()

In [ ]:
# Replace -999 with NaN
df_clean = df.replace(-999, np.nan)

# Check missing values
print("Missing values per column:")
print(df_clean.isna().sum())

# Count -999 values found
count_999 = (df == -999).sum().sum()
print(f"\nNumber of -999 values replaced: {count_999}")

In [ ]:
# Convert YEAR and DOY to datetime
df_clean['Date'] = pd.to_datetime(
    df_clean['YEAR'].astype(str) + df_clean['DOY'].astype(str).str.zfill(3),
    format='%Y%j'
)
df_clean['Month'] = df_clean['Date'].dt.month
df_clean['Year'] = df_clean['Date'].dt.year

print(f"Date range: {df_clean['Date'].min()} to {df_clean['Date'].max()}")
print("\nSample of converted dates:")
df_clean[['YEAR', 'DOY', 'Date', 'Month', 'Year']].head()

In [ ]:
# Forward fill for missing values
df_clean = df_clean.fillna(method='ffill')

# Drop any remaining NAs
df_clean = df_clean.dropna()

print(f"Shape after cleaning: {df_clean.shape}")

# Save cleaned data
df_clean.to_csv('data/nigeria_clean.csv', index=False)
print("Cleaned data saved to: data/nigeria_clean.csv")

In [ ]:
# Monthly average temperature
monthly_temp = df_clean.groupby(['Year', 'Month'])['T2M'].mean().reset_index()
monthly_temp['Date'] = pd.to_datetime(monthly_temp['Year'].astype(str) + '-' + 
                                       monthly_temp['Month'].astype(str))

# Find warmest and coolest months
warmest = monthly_temp.loc[monthly_temp['T2M'].idxmax()]
coolest = monthly_temp.loc[monthly_temp['T2M'].idxmin()]

# Plot
plt.figure(figsize=(14, 6))
plt.plot(monthly_temp['Date'], monthly_temp['T2M'], 'orange', linewidth=1.5)

# Mark extremes
plt.scatter(warmest['Date'], warmest['T2M'], color='red', s=100, zorder=5)
plt.scatter(coolest['Date'], coolest['T2M'], color='blue', s=100, zorder=5)

# Add annotations
plt.annotate(f"{warmest['T2M']:.1f}°C\n{warmest['Date'].strftime('%b %Y')}", 
             xy=(warmest['Date'], warmest['T2M']), xytext=(10, 10),
             textcoords='offset points', fontsize=10, fontweight='bold')
plt.annotate(f"{coolest['T2M']:.1f}°C\n{coolest['Date'].strftime('%b %Y')}", 
             xy=(coolest['Date'], coolest['T2M']), xytext=(10, -20),
             textcoords='offset points', fontsize=10, fontweight='bold')

plt.title('Nigeria: Monthly Average Temperature (2015-2026)', fontsize=14, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Temperature (°C)', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n📊 Temperature Summary for Nigeria:")
print(f"   Average temperature: {df_clean['T2M'].mean():.1f}°C")
print(f"   Warmest month: {warmest['Date'].strftime('%B %Y')} - {warmest['T2M']:.1f}°C")
print(f"   Coolest month: {coolest['Date'].strftime('%B %Y')} - {coolest['T2M']:.1f}°C")

In [ ]:
# Monthly total precipitation
monthly_precip = df_clean.groupby(['Year', 'Month'])['PRECTOTCORR'].sum().reset_index()
monthly_precip['Date'] = pd.to_datetime(monthly_precip['Year'].astype(str) + '-' + 
                                         monthly_precip['Month'].astype(str))

# Find wettest month
wettest = monthly_precip.loc[monthly_precip['PRECTOTCORR'].idxmax()]

# Plot
plt.figure(figsize=(14, 6))
plt.bar(monthly_precip['Date'], monthly_precip['PRECTOTCORR'], 
        width=25, color='teal', alpha=0.7)

plt.axhline(y=monthly_precip['PRECTOTCORR'].mean(), color='red', linestyle='--', 
            label=f"Average: {monthly_precip['PRECTOTCORR'].mean():.1f} mm")

plt.title('Nigeria: Monthly Total Precipitation (2015-2026)', fontsize=14, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Total Precipitation (mm)', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n💧 Precipitation Summary for Nigeria:")
print(f"   Mean daily rainfall: {df_clean['PRECTOTCORR'].mean():.2f} mm/day")
print(f"   Wettest month: {wettest['Date'].strftime('%B %Y')} - {wettest['PRECTOTCORR']:.1f} mm")
print(f"   Dry days (%): {((df_clean['PRECTOTCORR'] < 1).sum() / len(df_clean)) * 100:.1f}%")

In [ ]:
# Distribution of daily precipitation
skewness = df_clean['PRECTOTCORR'].skew()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Normal scale
ax1.hist(df_clean['PRECTOTCORR'], bins=50, color='teal', edgecolor='black', alpha=0.7)
ax1.set_xlabel('Daily Precipitation (mm/day)')
ax1.set_ylabel('Frequency')
ax1.set_title('Normal Scale')
ax1.grid(True, alpha=0.3)

# Log scale
ax2.hist(df_clean['PRECTOTCORR'], bins=50, color='darkorange', edgecolor='black', alpha=0.7)
ax2.set_xlabel('Daily Precipitation (mm/day)')
ax2.set_ylabel('Frequency (log scale)')
ax2.set_title('Log Scale')
ax2.set_yscale('log')
ax2.grid(True, alpha=0.3)

plt.suptitle(f'Nigeria: Daily Precipitation Distribution (Skewness: {skewness:.2f})', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\n📊 Distribution Statistics:")
print(f"   Skewness: {skewness:.2f} (positive = many dry days, few very wet days)")
print(f"   Days with <1mm: {((df_clean['PRECTOTCORR'] < 1).sum())}")
print(f"   Days with >20mm: {((df_clean['PRECTOTCORR'] > 20).sum())}")

In [ ]:
# Correlation analysis
corr_cols = ['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M', 'WS2M']
corr_matrix = df_clean[corr_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', center=0, 
            square=True, fmt='.2f', linewidths=0.5)
plt.title('Nigeria: Climate Variable Correlations', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Find strongest correlations
corr_pairs = corr_matrix.unstack().sort_values(ascending=False)
corr_pairs = corr_pairs[corr_pairs < 0.999]

print("\n🔗 Strongest Correlations:")
for pair, value in corr_pairs.head(3).items():
    print(f"   {pair[0]} ↔ {pair[1]}: {value:.3f}")

print("\n📉 Strongest Negative Correlations:")
for pair, value in corr_pairs.tail(3).items():
    print(f"   {pair[0]} ↔ {pair[1]}: {value:.3f}")

In [ ]:
print("="*60)
print("NIGERIA CLIMATE SUMMARY (2015-2026)")
print("="*60)
print(f"🌡️  Mean Temperature: {df_clean['T2M'].mean():.1f}°C")
print(f"☀️  Mean Max Temperature: {df_clean['T2M_MAX'].mean():.1f}°C")
print(f"❄️  Mean Min Temperature: {df_clean['T2M_MIN'].mean():.1f}°C")
print(f"💧  Mean Daily Rainfall: {df_clean['PRECTOTCORR'].mean():.2f} mm")
print(f"🏜️  Dry Days: {((df_clean['PRECTOTCORR'] < 1).sum() / len(df_clean)) * 100:.1f}%")
print(f"🌊  Max Daily Rainfall: {df_clean['PRECTOTCORR'].max():.1f} mm")
print(f"📅  Data Period: {df_clean['Date'].min().date()} to {df_clean['Date'].max().date()}")